# Week 10 — Wednesday: Ridge and Lasso

**DATA 202 · Calvin University**

Monday's line priced Ames houses from 47 columns, with no overfitting. Today we make a model that **does** overfit, and then take some of its freedom back.

> 🎯 **When a curve fits our houses too well, how do we tame it?**

**Today's plan (~40 min):**

| Time | Part |
|---|---|
| ~10 min | 1 · Thirty houses, one bendy curve |
| ~5 min | 2 · Trillion-dollar slopes |
| ~8 min | 3 · Ridge: a price on big slopes (SLO 10B) |
| ~7 min | 4 · Choosing alpha (SLO 10B) |
| ~10 min | 5 · Lasso: some slopes go to zero (SLO 10B) |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: De Cock (2011), *Journal of Statistics Education* 19(3); from the Ames City Assessor's Office</sub>

---
## 1 · Thirty Houses, One Bendy Curve · ~10 min

* Suppose we only had **30 sales** to learn from, picked at random from Monday's training houses
* Model A: Monday's **straight line** on living area
* Model B: a **degree-10 curve**. `PolynomialFeatures(10)` turns `living_area` into 10 columns: area, area², …, area¹⁰. A line on those 10 columns is a curve on the plot

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import root_mean_squared_error

ames = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/ames.csv")
X = ames.drop(columns="price")
y = ames["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)    # Monday's split

few = X_train.sample(30, random_state=29)          # pretend these 30 are all the sales we have
few_price = y_train[few.index]
print(f"30 houses, from {few['living_area'].min()} to {few['living_area'].max()} sq ft")

In [ ]:
line = LinearRegression().fit(few[["living_area"]], few_price)

curve = make_pipeline(PolynomialFeatures(10, include_bias=False),     # area, area², ..., area¹⁰
                      StandardScaler(),
                      LinearRegression())
curve.fit(few[["living_area"]], few_price)

sizes = pd.DataFrame({"living_area": np.linspace(808, 2772, 300)})    # from the smallest to the largest of the 30
fig = px.scatter(x=few["living_area"], y=few_price, labels={"x": "living area (sq ft)", "y": "sale price ($)"},
                 title="30 Houses: a Straight Line and a Degree-10 Curve")
fig.add_scatter(x=sizes["living_area"], y=line.predict(sizes), name="straight line")
fig.add_scatter(x=sizes["living_area"], y=curve.predict(sizes), name="degree-10 curve")
fig.update_yaxes(range=[0, 600_000])
fig.show()

> 📄 **Handout 1 — Thirty houses.** 🎯 Which model misses less on these 30 houses? And on *other* houses of the same sizes? Circle where the curve follows one house instead of the trend.

In [ ]:
others = X_train.drop(few.index)                                    # the other 2,163 training houses
others = others[others["living_area"].between(808, 2772)]           # only the sizes the 30 cover
others_price = y_train[others.index]

fig.add_scatter(x=others["living_area"], y=others_price, mode="markers", name="other houses",
                marker=dict(color="gray", size=4, opacity=0.3))
fig.show()

for name, model in [("straight line", line), ("degree-10 curve", curve)]:
    on_30 = root_mean_squared_error(few_price, model.predict(few[["living_area"]]))
    on_others = root_mean_squared_error(others_price, model.predict(others[["living_area"]]))
    print(f"{name:15} | RMSE on the 30: ${on_30:,.0f} | on {len(others):,} other houses: ${on_others:,.0f}")

<details><summary>Answer</summary>

* **On the 30:** the curve misses by **$20,557**, the line by $39,592. The curve wins
* **On 2,055 other houses:** the curve misses by **$205,415**, the line by $62,915. The curve loses badly: **overfitting**
* The curve chases single houses: **$534,000** at 2,200 sq ft, then **$220,000** at 2,450 sq ft. A bigger house for less than half the price
* Same story as kNN with k = 1 and a tree with no depth limit: it memorized the houses it learned from
</details>

---
## 2 · Trillion-Dollar Slopes · ~5 min

The curve is still a line, on 10 columns. Its slopes:

In [ ]:
# curve[0] is the pipeline's first step (PolynomialFeatures), curve[-1] its last (LinearRegression)
pd.Series(curve[-1].coef_ / 1e9, index=curve[0].get_feature_names_out(), name="slope (billions of $)").round(1)

> 💬 **Question:** Slopes of trillions of dollars, half of them negative. Yet the curve still passes near the 30 houses. How?

<details><summary>Answer</summary>

* area, area², …, area¹⁰ all rise together: they are near-copies of one column
* So the slopes play **tug of war**: trillions of dollars on one column, minus trillions on the next. On the 30 houses they almost cancel
* Between and beyond those houses the canceling slips: the wiggles. (Reading 8's abalone did the same, at −20.8 and +20.6)
* Picture the squared error as a landscape over two slopes: a long flat valley, with no single bottom
</details>

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/10/images/ridge_surface.png" width="640" alt="Two 3-D surfaces of squared error over two coefficients, beta1 and beta2. Left, least squares: a long flat trough, labeled unstable. Right, ridge: a bowl with a single lowest point, labeled stable.">

---
## 3 · Ridge: a Price on Big Slopes (SLO 10B) · ~8 min

**Ridge** adds a cost for every slope's size to the least-squares error:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \sum b_j^2$$

* `alpha` = 0: plain least squares, Part 1's curve
* A bigger `alpha` pushes every slope toward 0. A tug of war at ±trillions now costs a fortune
* The valley gets a bottom (right picture): one best set of slopes
* The penalty compares slopes with each other, so the columns must share one ruler: **scale first**

> 📄 **Handout 2 — Three alphas.** Label each curve: overfitting, about right, or underfitting. 🗣️ Then talk with a neighbor.

In [ ]:
fig = px.scatter(x=few["living_area"], y=few_price, labels={"x": "living area (sq ft)", "y": "sale price ($)"},
                 title="Ridge on the 30 Houses, Three Alphas")
for alpha in [0, 10, 1000]:
    ridge = make_pipeline(PolynomialFeatures(10, include_bias=False), StandardScaler(), Ridge(alpha=alpha))
    ridge.fit(few[["living_area"]], few_price)
    fig.add_scatter(x=sizes["living_area"], y=ridge.predict(sizes), name=f"alpha = {alpha}")
fig.update_yaxes(range=[0, 600_000])
fig.show()

<details><summary>Answer</summary>

* **alpha = 0:** Part 1's wiggles: **overfitting**
* **alpha = 10:** a gentle upward bend. Bigger houses add more per square foot, which a straight line can't show: **about right**
* **alpha = 1,000:** almost flat. The penalty squashes every slope, so it misses even the 30 houses: **underfitting**
* 🗣️ With 30 different houses, the alpha = 0 curve would change the most: it follows each house. The alpha = 1,000 curve would barely move, but it is wrong everywhere
</details>

In [ ]:
ridge = make_pipeline(PolynomialFeatures(10, include_bias=False), StandardScaler(), Ridge(alpha=10))
ridge.fit(few[["living_area"]], few_price)
pd.Series(ridge[-1].coef_, index=ridge[0].get_feature_names_out(), name="slope ($)").round(0)

<details><summary>Answer</summary>

* Slopes of **$20,251** and smaller, instead of trillions. No more tug of war
* Same 10 columns, all still in the model, just small
</details>

---
## 4 · Choosing alpha (SLO 10B) · ~7 min

`alpha` is a knob, like `k` and `max_depth`. Choose it with cross-validation, on the 30 houses only.

(`ridge__alpha` means "the `alpha` of the step named `ridge`": `make_pipeline` names each step after its class.)

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
search = GridSearchCV(make_pipeline(PolynomialFeatures(10, include_bias=False), StandardScaler(), Ridge()),
                      {"ridge__alpha": alphas}, cv=5, scoring="neg_root_mean_squared_error", return_train_score=True)
search.fit(few[["living_area"]], few_price)

results = pd.DataFrame({"alpha": alphas,
                        "folds it learned from": -search.cv_results_["mean_train_score"],
                        "held-out fold": -search.cv_results_["mean_test_score"]})
fig = px.line(results, x="alpha", y=["folds it learned from", "held-out fold"], markers=True, log_x=True,
              labels={"value": "RMSE ($)", "variable": "scored on"}, title="Ridge: Choosing alpha by 5-Fold CV")
fig.show()

> 📄 **Handout 3 — Choose alpha.** Circle the best alpha. Label the overfitting side and the underfitting side.

<details><summary>Answer</summary>

* **Best: alpha = 10**, held-out RMSE **$38,390**
* **Small alpha:** about $33,000 on the folds it learned from, up to $120,000 on the held-out fold. A huge gap: **overfitting**
* **alpha = 1,000:** about $80,000 on both: **underfitting**
* This time a **larger** knob means a **simpler** model, like k in kNN. For trees, a larger `max_depth` meant a more complex one
</details>

Does CV's choice hold up on the other houses?

In [ ]:
best = search.best_estimator_
for name, model in [("straight line", line), ("degree-10 curve", curve), ("Ridge, alpha = 10", best)]:
    on_others = root_mean_squared_error(others_price, model.predict(others[["living_area"]]))
    print(f"{name:17} | RMSE on {len(others):,} other houses: ${on_others:,.0f}")

<details><summary>Answer</summary>

* Ridge: **$58,426**, better than the straight line ($62,915) and far better than the wiggly curve ($205,415)
* Ridge kept the curve's real bend and dropped its wiggles
</details>

> 📌 **One sample:** each CV fold here holds only 6 houses. Another 30 houses could pick another alpha. With so little data, the best cure is more houses.

---
## 5 · Lasso: Some Slopes Go to Zero (SLO 10B) · ~10 min

**Lasso** charges for the slopes' absolute sizes instead:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \sum |b_j|$$

* Same idea, one difference: Lasso pushes many slopes **all the way to 0**. Those columns drop out of the model
* Most useful with many columns. Back to all 2,193 training houses and Monday's 47 columns, now scaled

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric = [c for c in X.columns if c not in ("neighborhood", "year_sold")]
lasso = Pipeline([
    ("prep", ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), numeric),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["neighborhood"]),
    ])),
    ("lasso", Lasso(max_iter=100_000)),
])

rows = []
for alpha in [10, 100, 300, 1000, 3000, 10000, 30000]:
    lasso.set_params(lasso__alpha=alpha)
    cv_rmse = -cross_val_score(lasso, X_train, y_train, cv=5, scoring="neg_root_mean_squared_error").mean()
    kept = (lasso.fit(X_train, y_train)[-1].coef_ != 0).sum()
    rows.append({"alpha": alpha, "columns kept (of 47)": kept, "CV RMSE ($)": round(cv_rmse)})
pd.DataFrame(rows)

> 📄 **Handout 4 — What Lasso keeps.** Monday's line with all 47 columns: CV RMSE **$27,876**. Find the simplest model within $1,000 of it. Which row is underfitting?

<details><summary>Answer</summary>

* **alpha = 300:** **25** of 47 columns, $28,563: only $687 worse than Monday's model, with about half the columns
* **alpha = 30,000:** 4 columns, $51,003: **underfitting**
* No alpha beats Monday by much: Monday's line wasn't overfitting. Here Lasso's job is **choosing columns**
</details>

In [ ]:
lasso.set_params(lasso__alpha=300).fit(X_train, y_train)
slopes = pd.Series(lasso[-1].coef_, index=lasso[0].get_feature_names_out())
slopes.index = slopes.index.str.replace("num__|cat__", "", regex=True)     # drop the step-name prefixes
print("kept:   ", ", ".join(slopes[slopes != 0].index))
print("dropped:", ", ".join(slopes[slopes == 0].index))

<details><summary>Answer</summary>

* Dropped: `second_floor_area` (it's already inside `living_area`), `half_baths`, `garage_cars` (`garage_area` stays), and many neighborhoods
* With every penalty, Lasso decides which columns matter. We didn't
</details>

> "By inciting organizations to treat all data as useful, and by developing tools with a hitherto unmatched ability to find patterns, software engineers have created tools endowed with new and somewhat alien powers."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Lasso kept some neighborhoods and dropped others. If an assessor's office valued your home this way, what would you want to know about the columns it kept and dropped?

<details><summary>Answer</summary>

* **Which columns could go in at all:** the neighborhood is in. In many U.S. cities, neighborhood lines still follow old redlining maps, so a neighborhood slope can carry that history into a price
* **Why a column was dropped:** "Lasso set it to 0" isn't a reason a homeowner can contest; with another split, other columns might survive
* Penalties tame wild slopes. They don't decide what *should* count in a home's value. People do
</details>

---
### Wrap-up

1. A bendy curve on few houses has too much freedom → it **overfits**: wiggles, and trillion-dollar slopes that cancel
2. **Ridge** charges α × Σ b²: every slope shrinks, none goes to 0. The wiggles go, the real bend stays
3. **Lasso** charges α × Σ |b|: many slopes go to exactly 0, so it also chooses columns
4. Choose α with **cross-validation**; scale first; small α overfits, large α underfits 📌